# OfficeAgentEnv - Layer 2 PPO Training (Colab)

This notebook trains a policy with PPO using **live environment interaction** (no static dataset, no heuristic policy).

## Sections
1. Setup
2. Model loading
3. Environment integration
4. Helpers (observation formatting + action parsing)
5. PPO setup
6. Baseline evaluation
7. PPO training loop
8. Reward plot
9. Post-training evaluation
10. Before vs After
11. Save model


In [ ]:
!pip install -q transformers trl accelerate bitsandbytes matplotlib

: 

In [ ]:
import json
import os
import random
import time
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer

# Expected uploaded folders in Colab runtime:
# /content/env, /content/graders
import sys
sys.path.append("/content")

from env.environment import ExecAssistEnv
from env.models import ActionType, EmailCategory, ExecAssistAction

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


In [ ]:
# 5B-class on T4 is fragile; default to 3B for stability.
# You can try a larger model only if memory allows.
model_name = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLMWithValueHead.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("Model loaded:", model_name)


In [ ]:
MAX_STEPS = {"easy": 10, "medium": 15, "hard": 12}
TASKS = ["easy", "medium", "hard"]

SYSTEM_PROMPT = """
You are an enterprise executive-assistant RL policy.
Return exactly one JSON action for the current observation.
Valid action_type values:
- classify_email
- reply_email
- schedule_meeting
- ignore_email
- assign_task
- query_status
- update_project
Always include a valid email_id from pending_emails.
For classify_email include category.
For reply_email include reply_text.
For schedule_meeting include meeting_start_time and meeting_end_time.
For assign_task include team.
For update_project include project_id and project_status.
Output JSON only.
""".strip()


def format_observation_as_text(obs: Dict[str, Any]) -> str:
    pending = obs.get("pending_emails", [])
    calendar = obs.get("calendar_events", [])
    world_state = obs.get("world_state", {})
    compact_obs = {
        "task_name": obs.get("task_name"),
        "current_step": obs.get("current_step"),
        "last_action_result": obs.get("last_action_result", ""),
        "pending_emails": [
            {
                "email_id": e.get("email_id"),
                "sender": e.get("sender"),
                "subject": e.get("subject"),
                "body": str(e.get("body", ""))[:180],
            }
            for e in pending
        ],
        "calendar_events": [
            {
                "title": c.get("title"),
                "start_time": c.get("start_time"),
                "end_time": c.get("end_time"),
            }
            for c in calendar
        ],
        "world_state": {
            "projects": world_state.get("projects", []),
            "team_load": world_state.get("team_load", {}),
        },
    }
    return json.dumps(compact_obs, ensure_ascii=True)


def _extract_json(text: str) -> Dict[str, Any]:
    cleaned = text.replace("```json", "").replace("```", "").strip()
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start < 0 or end < 0 or end <= start:
        raise ValueError("No JSON object found")
    return json.loads(cleaned[start : end + 1])


def parse_action(action_text: str, obs: Dict[str, Any]) -> Dict[str, Any]:
    action = _extract_json(action_text)
    pending = obs.get("pending_emails", [])
    valid_ids = {e.get("email_id") for e in pending}

    valid_types = {
        ActionType.CLASSIFY_EMAIL.value,
        ActionType.REPLY_EMAIL.value,
        ActionType.SCHEDULE_MEETING.value,
        ActionType.IGNORE_EMAIL.value,
        ActionType.ASSIGN_TASK.value,
        ActionType.QUERY_STATUS.value,
        ActionType.UPDATE_PROJECT.value,
    }

    action_type = action.get("action_type")
    email_id = action.get("email_id")
    if action_type not in valid_types:
        raise ValueError("Invalid action_type")
    if email_id not in valid_ids:
        raise ValueError("Invalid email_id")

    if action_type == ActionType.CLASSIFY_EMAIL.value:
        allowed = {c.value for c in EmailCategory}
        if action.get("category") not in allowed:
            raise ValueError("classify_email missing/invalid category")

    if action_type == ActionType.REPLY_EMAIL.value:
        if not action.get("reply_text"):
            action["reply_text"] = "Thanks for your message. I will follow up shortly."

    if action_type == ActionType.SCHEDULE_MEETING.value:
        action.setdefault("meeting_title", "Meeting")
        action.setdefault("meeting_start_time", "2024-07-01 11:00")
        action.setdefault("meeting_end_time", "2024-07-01 11:30")
        if not action.get("participants"):
            sender = next((e.get("sender") for e in pending if e.get("email_id") == email_id), "unknown@example.com")
            action["participants"] = [sender]

    if action_type == ActionType.ASSIGN_TASK.value:
        action.setdefault("team", "engineering")

    if action_type == ActionType.UPDATE_PROJECT.value:
        action.setdefault("project_id", "P1")
        action.setdefault("project_status", "on_track")

    return action


In [ ]:
ppo_config = PPOConfig(
    learning_rate=1e-5,
    batch_size=2,
    mini_batch_size=1,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    tokenizer=tokenizer,
)


def generate_action_text(obs_text: str, temperature: float = 0.8, max_new_tokens: int = 180) -> str:
    prompt = f"{SYSTEM_PROMPT}\n\nObservation:\n{obs_text}\n\nAction JSON:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536).to(model.pretrained_model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def generate_and_parse_action(obs: Dict[str, Any], retry_once: bool = True) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    obs_text = format_observation_as_text(obs)
    try:
        text = generate_action_text(obs_text)
        action = parse_action(text, obs)
        return action, text
    except Exception:
        if retry_once:
            try:
                text = generate_action_text(obs_text)
                action = parse_action(text, obs)
                return action, text
            except Exception:
                return None, None
        return None, None


In [ ]:
def run_episode(task: str, seed: int, train: bool) -> Tuple[float, List[Dict[str, Any]]]:
    env = ExecAssistEnv(task_name=task, seed=seed)
    obs = env.reset().model_dump()
    done = False
    total_reward = 0.0
    episode_steps: List[Dict[str, Any]] = []
    safety_counter = 0

    while not done and safety_counter < (MAX_STEPS[task] * 3):
        safety_counter += 1
        obs_text = format_observation_as_text(obs)

        action, action_text = generate_and_parse_action(obs, retry_once=True)
        if action is None or action_text is None:
            # safe skip on repeated parse failure
            episode_steps.append(
                {
                    "observation": obs_text,
                    "action": "<parse_failed>",
                    "reward": 0.0,
                }
            )
            print(f"Episode parse-failure skip | task={task} | seed={seed}")
            break

        step_result = env.step(ExecAssistAction(**action)).model_dump()
        reward = float(step_result.get("reward", 0.0))
        done = bool(step_result.get("done", False))

        if train:
            ppo_trainer.step([obs_text], [action_text], [reward])

        episode_steps.append(
            {
                "observation": obs_text,
                "action": action_text,
                "reward": reward,
            }
        )

        total_reward += reward
        obs = step_result["observation"]

    return total_reward, episode_steps


def evaluate_policy(num_episodes: int = 4) -> Tuple[float, List[float]]:
    rewards = []
    for i in range(num_episodes):
        task = random.choice(TASKS)
        seed = int(time.time()) + i + random.randint(1, 9999)
        ep_reward, _ = run_episode(task=task, seed=seed, train=False)
        rewards.append(ep_reward)
        print(f"[BASELINE EVAL] Episode {i+1}/{num_episodes} | task={task} | reward={ep_reward:.4f}")
    avg_reward = sum(rewards) / max(1, len(rewards))
    return avg_reward, rewards


BASELINE_EPISODES = 4
TRAIN_EPISODES = 20
POST_EVAL_EPISODES = 4

print("=== Baseline Evaluation (No PPO updates) ===")
before_avg_reward, baseline_rewards = evaluate_policy(num_episodes=BASELINE_EPISODES)
print(f"Baseline Avg Reward: {before_avg_reward:.4f}")

print("\n=== PPO Training ===")
reward_history: List[float] = []
trajectories: List[Dict[str, Any]] = []
for ep in range(1, TRAIN_EPISODES + 1):
    task = random.choice(TASKS)
    seed = int(time.time()) + ep + random.randint(1, 9999)
    ep_reward, steps = run_episode(task=task, seed=seed, train=True)
    reward_history.append(ep_reward)
    trajectories.append(
        {
            "episode": ep,
            "task": task,
            "seed": seed,
            "reward": ep_reward,
            "steps": steps,
        }
    )
    print(f"[TRAIN] Episode {ep}/{TRAIN_EPISODES} | task={task} | reward={ep_reward:.4f}")

print("\n=== Post-Training Evaluation (No PPO updates) ===")
after_avg_reward, post_rewards = evaluate_policy(num_episodes=POST_EVAL_EPISODES)
print(f"Final Avg Reward: {after_avg_reward:.4f}")

print("\n=== Before vs After ===")
print(f"Before: {before_avg_reward:.4f}")
print(f"After:  {after_avg_reward:.4f}")

if after_avg_reward <= before_avg_reward:
    print("Note: Improvement is not guaranteed in short runs. Try more training episodes or a smaller model.")


In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(range(1, len(reward_history) + 1), reward_history, label="Train Episode Reward", marker="o", linewidth=1.2)
plt.axhline(before_avg_reward, color="tab:orange", linestyle="--", label=f"Baseline Avg={before_avg_reward:.3f}")
plt.axhline(after_avg_reward, color="tab:green", linestyle="--", label=f"Post Avg={after_avg_reward:.3f}")
plt.title("OfficeAgentEnv PPO Reward Trend")
plt.xlabel("Episode")
plt.ylabel("Cumulative Reward")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

save_dir = "/content/officeagent_rl_model/"
os.makedirs(save_dir, exist_ok=True)

# Save full policy + tokenizer
model.pretrained_model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Saved full model + tokenizer to: {save_dir}")
